# SVI slice fit — raw points vs fitted slice

Visual demonstration for operator review (blueprint step 9 acceptance: "visualize the raw
points versus fitted slices"). This file is a notebook (percent `# %%` cells — open it in
VS Code / Jupyter, or run it as a script). It ONLY calls the tested surface engine and plots;
no calibration logic lives here.

Run: `uv run --group notebooks python notebooks/demo_surface_fit.py`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from algotrading.infra.surfaces import calibrate_slice, svi_total_variance

A ground-truth SVI smile, observed as noisy market total-variance points for one maturity.

In [ ]:
maturity = 0.5
true_params = dict(a=0.025, b=0.15, rho=-0.45, m=0.0, sigma=0.12)
k = np.linspace(-0.4, 0.4, 15)
rng = np.random.default_rng(0)  # fixed seed -> reproducible demo
w_market = svi_total_variance(k, **true_params) + rng.normal(0.0, 5e-4, size=k.size)

Calibrate the slice with the tested engine and inspect the fit diagnostics.

In [ ]:
fit = calibrate_slice(k, w_market)
print(f"converged={fit.converged}  rmse={fit.rmse:.2e}  bound_hits={fit.bound_hits}")
print(f"params: {fit.params}")

Plot total variance and the implied-vol smile: market points vs the fitted slice.

In [ ]:
k_fine = np.linspace(float(k.min()), float(k.max()), 200)
w_fit = fit.params.total_variance(k_fine)

fig, (ax_w, ax_iv) = plt.subplots(1, 2, figsize=(11, 4))
ax_w.scatter(k, w_market, color="C1", zorder=3, label="market")
ax_w.plot(k_fine, w_fit, color="C0", label="SVI fit")
ax_w.set(xlabel="log-moneyness k", ylabel="total variance w", title="SVI slice fit")
ax_w.legend()

ax_iv.scatter(k, np.sqrt(w_market / maturity), color="C1", zorder=3, label="market")
ax_iv.plot(k_fine, np.sqrt(w_fit / maturity), color="C0", label="SVI fit")
ax_iv.set(xlabel="log-moneyness k", ylabel="implied vol", title="Implied-vol smile")
ax_iv.legend()
fig.tight_layout()
plt.show()